---
title: Building C Seismic Design
project: Beaverton Creak Apartments
location: Beaverton, OR
client: GBD Architects
job_number: 10022500446
author:
  name: Andy Sazima
  initials: AGS
  email: andy.sazima@kpff.com
---

In [1]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

import pandas as pd
import sympy as sp
from structural_tools.notebook.calculation import check_value, set_params_columns
from structural_tools.notebook.calculation import feet_inches as fi
from structural_tools.notebook.display import (
    display_table,
    display_text,
    sig_figs,  # noqa: F401
)
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_floor = 35 / 1000 * ksf
# U_pavers = 15 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
# Plan dimensions
l_C1__C2 = fi(31, 4)
l_C2__C3 = fi(31, 5)
l_C3__C4 = fi(26, 5)
l_C4__C5 = fi(17, 11)
#
l_C6__C7 = fi(21, 4)
l_C7__C8 = fi(26, 6)
l_C8__C9 = fi(36, 5, 1 / 2)

l_C1__C21 = fi(35, 1, 1 / 2)
l_C21__C29 = fi(24, 8)
l_C29__C42 = fi(34, 4)
l_C42__C55 = fi(24, 8)
#
l_C59__C7 = fi(21, 4)
l_C7__C83 = fi(36, 10)
l_C83__C9 = fi(26, 1, 1 / 2)

l_C3__C33 = fi(8, 10)
l_C33__C4 = l_C3__C4 - l_C3__C33
l_C7__C73 = fi(7, 7, 1 / 4)
l_C6__C73 = l_C6__C7 + l_C7__C73
l_C73__C8 = l_C7__C8 - l_C7__C73
l_C8__C81 = fi(1, 5, 0.75)
l_C81__C83 = fi(8, 10)
l_C7__C81 = l_C7__C8 + l_C8__C81

l_CA__CB = fi(30, 9)
l_CB__CC = fi(5, 6)
l_CC__CD = fi(30, 9)

l_CW__CX = fi(30, 9)
l_CX__CY = fi(5, 6)
l_CY__CZ = fi(30, 9)

l_CW__CW6 = fi(19, 11, 0.25)
l_CW6__CX = l_CW__CX - l_CW__CW6
l_CY__CY6 = fi(19, 8, 0.25)
l_CY6__CZ = l_CY__CZ - l_CY__CY6

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 13.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10.5 * ft
h_wall__5 = 10.5 * ft

<IPython.core.display.Latex object>

In [7]:
P_wall = (
    7
    + fi(30, 9)
    + fi(79, 9, 1 / 2)
    + fi(86, 1, 1 / 4)
    + fi(30, 9)
    + fi(31, 4)
    + fi(36, 3)
    + fi(131, 3 / 4)
    + fi(100, 4, 3 / 4)
    + fi(36, 3)
) * ft

In [8]:
%%render
P_wall

<IPython.core.display.Latex object>

The areas of the building are:

In [9]:
%%render params
A_level = 13417 * ft**2
# A_pavers = 907 * ft**2
A_floor = A_level
# - A_pavers

<IPython.core.display.Latex object>

In [10]:
%%render
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall
A_wall__5 = h_wall__5 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [11]:
%%render
W_topfloor = U_floor * A_floor
# W_pavers = (U_floor + U_pavers) * A_pavers
W_floor__typ = U_floor * A_level

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4. The 5th (top) level has pavers in the N-E corner.

In [12]:
set_params_columns(4)

In [13]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_floor__5 = W_topfloor
# + W_pavers

<IPython.core.display.Latex object>

The roof weight is

In [14]:
%%render
W_roof = U_roof * A_floor

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [15]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__5 = U_wall * (A_wall__5 / 2 + A_wall__4 / 2)
W_wall__roof = U_wall * (A_wall__5 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [16]:
%%render 
W_level__2 = W_floor__2 + W_wall__2
W_level__3 = W_floor__3 + W_wall__3
W_level__4 = W_floor__4 + W_wall__4
W_level__5 = W_floor__5 + W_wall__5
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [17]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__5 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [18]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [19]:
set_params_columns(5)

In [20]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [21]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [22]:
set_params_columns(1)

In [23]:
%%render params
h_top__roof = fi(71, 3) * ft
h_bottom__roof = fi(58, 7) * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [24]:
%%render
h_n__check = check_value(h_n, 65 * ft, "<=")

<IPython.core.display.Latex object>

We will approximate the building period.

In [25]:
h_n = float(h_n)  # to prevent weirdness with units

In [26]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [27]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [28]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [29]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [30]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [31]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [32]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [33]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [34]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [35]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Level 1", "Level 2", "Level 3", "Level 4", "Level 5", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            float(h_wall__5),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__5),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [36]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [37]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,371.315313,55.5,20607.999844,0.245078,65.034920,0.000000,0.000000
Level 5,10.5,541.375625,45.0,24361.903125,0.289721,76.881523,65.034920,682.866656
Level 4,10.5,541.375625,34.5,18677.459063,0.222120,58.942501,141.916443,2172.989304
Level 3,10.5,541.375625,24.0,12993.015000,0.154518,41.003479,200.858944,4282.008211
Level 2,10.5,551.630000,13.5,7447.005000,0.088563,23.501328,241.862422,6821.563646
Level 1,13.5,0.000000,0.0,0.000000,0.000000,0.000000,265.363751,10403.974282


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [38]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Level 5", "Level 4", "Level 3", "Level 2", "Level 1"]
walls = {
    # N-S
    "C-1.7": "N-S",
    "C-2.1": "N-S",
    "C-2.9": "N-S",
    "C-3": "N-S",
    "C-3.3": "N-S",
    "C-4": "N-S",
    "C-4.2": "N-S",
    "C-5": "N-S",
    "C-5.5": "N-S",
    "C-5.9": "N-S",
    "C-6": "N-S",
    "C-7--C-W": "N-S",
    "C-7--C-Y": "N-S",
    "C-7.3": "N-S",
    "C-7.4": "N-S",
    "C-7.9": "N-S",
    # "C-8--C-W": "N-S",
    "C-8--C-W.6": "N-S",
    "C-8.1": "N-S",
    "C-8.3": "N-S",
    # E-W
    "C-A.5--C-2": "E-W",
    "C-B--C-2": "E-W",
    "C-B--C-4": "E-W",
    "C-B--C-4.9": "E-W",
    "C-W.6--C-7.3": "E-W",
    # "C-X--C-5.8": "E-W",
    "C-X--C-6.5": "E-W",
    "C-X--C-8.5": "E-W",
    "C-C--C-2.5": "E-W",
    "C-C--C-3.6": "E-W",
    "C-C--C-5.2": "E-W",
    # "C-Y--C-5.8": "E-W",
    "C-Y--C-6.5": "E-W",
    "C-Y--C-7.2": "E-W",
    "C-Y.3--C-8": "E-W",
    "C-Y.6--C-8": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (paver weight vs. floor weight, for instance), but the vast majority of the area of each level is typical floor weight.

Due to this building's geometry, we must include an additional 

$$
f_{p,x} = F_x / A_{level}
$$

In [39]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__5 = seismic_loads.loc["Level 5", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [40]:
from structural_tools.seismic.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [41]:
d_trib_ns = 67 / 2 * ft

In [42]:
# Wall tributary widths and lengths
t_w__C17 = fi(29, 9)
t_w__C21 = fi(18, 0)
# t_w__C21 = l_C1__C21 + (l_C21__C29 / 2)
t_w__C29 = (l_C21__C29 / 2) + (l_C29__C42 / 2)
t_w__C42 = (l_C29__C42 / 2) + (l_C42__C55 / 2)
t_w__C55 = (l_C42__C55 / 2) + 0  # placeholder for angled area that will be manualled added

t_w__C3 = l_C2__C3 + (l_C3__C33 / 2)
t_w__C33 = (l_C3__C33 / 2) + (l_C33__C4 / 2)
t_w__C4 = (l_C33__C4 / 2) + (l_C4__C5 / 2)
t_w__C5 = (l_C4__C5 / 2) + 0  # placeholder for angled area that will be manualled added

t_w__C59 = (l_C59__C7 / 2) + 0  # placeholder for angled area that will be manualled added
# t_w__C7 = (l_C59__C7 / 2) + (l_C7__C81 / 2)
t_w__C7 = fi(15, 5)
t_w__C74 = fi(13, 0)
t_w__C79 = fi(9, 1)
# t_w__C81 = (l_C7__C81 / 2) + (l_C81__C83 / 2)
t_w__C81 = fi(6, 1)
t_w__C83 = (l_C81__C83 / 2) + l_C83__C9

t_w__C6 = (l_C6__C7 / 2) + 0  # placeholder for angled area that will be manualled added
t_w__C7__1 = (l_C6__C7 / 2) + (l_C7__C73 / 2)
t_w__C73 = (l_C7__C73 / 2) + (l_C73__C8 / 2)
t_w__C8 = (l_C73__C8 / 2) + l_C8__C9 - 7  # 7 foot jog

t_l__C17 = float(d_trib_ns)
t_l__C21 = float(d_trib_ns)
t_l__C29 = float(d_trib_ns)
t_l__C3 = float(d_trib_ns)
t_l__C33 = float(d_trib_ns)
t_l__C4 = float(d_trib_ns)
t_l__C42 = float(d_trib_ns)
t_l__C5 = float(d_trib_ns)
t_l__C55 = float(d_trib_ns)
t_l__C59 = float(d_trib_ns)
t_l__C6 = float(d_trib_ns)
t_l__C7 = float(d_trib_ns)
t_l__C73 = float(d_trib_ns)
t_l__C74 = float(d_trib_ns)
t_l__C79 = float(d_trib_ns)
t_l__C8__1 = fi(15, 8)
t_l__C8__2 = fi(17, 10)
t_l__C81 = float(d_trib_ns)
t_l__C83 = float(d_trib_ns)

In [43]:
flexible_ns.loc["Level 5", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__5)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 1", "tributary weight"] = float(f_p__2)

# Nonparallel irregularity
angle_degrees = 23
angle = math.radians(angle_degrees)
angled_walls = ["C-1.7", "C-2.1", "C-2.9", "C-3", "C-3.3", "C-4", "C-4.2", "C-5", "C-5.5"]
wall_indices = flexible_ns.index.get_level_values("Wall").isin(angled_walls)
flexible_ns.loc[wall_indices, "tributary weight"] *= math.cos(angle) + 0.3 * math.sin(angle)

# Shear wall height is for the floor below
flexible_ns.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
l_typ = 29
l_stairwall = 19
l_stairwall_short = fi(14, 3)
l_elevator = 11
wall_tributary = {
    "C-1.7": {"trib_width": float(t_w__C17), "trib_depth": t_l__C17, "wall_length": fi(20, 0)},
    "C-2.1": {"trib_width": float(t_w__C21), "trib_depth": t_l__C21, "wall_length": l_typ},
    "C-2.9": {"trib_width": float(t_w__C29), "trib_depth": t_l__C29, "wall_length": l_typ},
    "C-3": {"trib_width": float(t_w__C3), "trib_depth": t_l__C3, "wall_length": l_stairwall_short},
    "C-3.3": {"trib_width": float(t_w__C33), "trib_depth": t_l__C33, "wall_length": l_stairwall},
    "C-4": {"trib_width": float(t_w__C4), "trib_depth": t_l__C4, "wall_length": l_typ},
    "C-4.2": {"trib_width": float(t_w__C42), "trib_depth": t_l__C42, "wall_length": l_typ},
    "C-5": {"trib_width": float(t_w__C5), "trib_depth": t_l__C5, "wall_length": l_typ},
    "C-5.5": {"trib_width": float(t_w__C55), "trib_depth": t_l__C55, "wall_length": l_typ},
    "C-5.9": {"trib_width": float(t_w__C59), "trib_depth": t_l__C59, "wall_length": l_typ},
    "C-6": {"trib_width": float(t_w__C6), "trib_depth": t_l__C6, "wall_length": l_typ},
    "C-7--C-W": {"trib_width": float(t_w__C7__1), "trib_depth": t_l__C7, "wall_length": l_typ},
    "C-7--C-Y": {"trib_width": float(t_w__C7), "trib_depth": t_l__C7, "wall_length": l_typ},
    "C-7.3": {"trib_width": float(t_w__C73), "trib_depth": t_l__C73, "wall_length": l_elevator},
    "C-7.4": {"trib_width": float(t_w__C74), "trib_depth": t_l__C74, "wall_length": fi(10, 10)},
    "C-7.9": {"trib_width": float(t_w__C79), "trib_depth": t_l__C79, "wall_length": fi(7, 7)},
    # "C-8--C-W": {"trib_width": float(t_w__C8), "trib_depth": t_l__C8__1, "wall_length": fi(9, 6)},
    "C-8--C-W.6": {
        "trib_width": float(t_w__C8),
        "trib_depth": t_l__C8__2 + t_l__C8__1,
        "wall_length": l_elevator + 6.5,
    },
    "C-8.1": {"trib_width": float(t_w__C81), "trib_depth": t_l__C81, "wall_length": l_stairwall_short},
    "C-8.3": {"trib_width": float(t_w__C83), "trib_depth": t_l__C83, "wall_length": l_stairwall},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map({
    key: value["wall_length"] for key, value in wall_tributary.items()
})
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map({
    key: value["trib_width"] for key, value in wall_tributary.items()
})
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map({
    key: value["trib_depth"] for key, value in wall_tributary.items()
})
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-2.1", "tributary area"] += 100
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5", "tributary area"] += 435.8
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5.5", "tributary area"] += 290
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5.9", "tributary area"] += 416
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-6", "tributary area"] += 195
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-8.3", "tributary area"] += 23.5

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [44]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [45]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [46]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [47]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [48]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [49]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [50]:
display_table(
    dataframe=flexible_ns,
    levels="Level 5",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 5",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 1",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 1",
    label="tab:shear_wall_forces_level1_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-1.7,5.030060,996.625000,5013.083134,5013.083134,20.000000,250.654157,175.457910,310.714286,1,6
C-2.1,5.030060,703.000000,3536.131889,3536.131889,29.000000,121.935582,85.354908,310.714286,1,6
C-2.9,5.030060,988.250000,4970.956385,4970.956385,29.000000,171.412289,119.988602,310.714286,1,6
C-3,5.030060,1200.416667,6038.167361,6038.167361,14.250000,423.731043,296.611730,310.714286,1,6
C-3.3,5.030060,442.479167,2225.696574,2225.696574,19.000000,117.141925,81.999347,310.714286,1,6
C-4,5.030060,594.625000,2990.999181,2990.999181,29.000000,103.137903,72.196532,310.714286,1,6
C-4.2,5.030060,988.250000,4970.956385,4970.956385,29.000000,171.412289,119.988602,310.714286,1,6
C-5,5.030060,735.904167,3701.641808,3701.641808,29.000000,127.642821,89.349975,310.714286,1,6
C-5.5,5.030060,703.166667,3536.970232,3536.970232,29.000000,121.964491,85.375144,310.714286,1,6


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-1.7,5.946323,996.625000,5926.254202,10939.337337,20.000000,546.966867,382.876807,460.714286,1,4
C-2.1,5.946323,703.000000,4180.265099,7716.396988,29.000000,266.082655,186.257858,310.714286,1,6
C-2.9,5.946323,988.250000,5876.453747,10847.410132,29.000000,374.048625,261.834038,310.714286,1,6
C-3,5.946323,1200.416667,7138.065286,13176.232647,14.250000,924.647905,647.253534,769.642857,1,2
C-3.3,5.946323,442.479167,2631.124065,4856.820638,19.000000,255.622139,178.935497,310.714286,1,6
C-4,5.946323,594.625000,3535.832339,6526.831520,29.000000,225.063156,157.544209,310.714286,1,6
C-4.2,5.946323,988.250000,5876.453747,10847.410132,29.000000,374.048625,261.834038,310.714286,1,6
C-5,5.946323,735.904167,4375.923903,8077.565711,29.000000,278.536749,194.975724,310.714286,1,6
C-5.5,5.946323,703.166667,4181.256153,7718.226385,29.000000,266.145737,186.302016,310.714286,1,6


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-1.7,4.558848,996.625000,4543.461555,15482.798892,20.000000,774.139945,541.897961,600.000000,1,3
C-2.1,4.558848,703.000000,3204.869909,10921.266897,29.000000,376.595410,263.616787,310.714286,1,6
C-2.9,4.558848,988.250000,4505.281206,15352.691338,29.000000,529.403150,370.582205,460.714286,1,4
C-3,4.558848,1200.416667,5472.516719,18648.749366,14.250000,1308.684166,916.078916,921.428571,2,4
C-3.3,4.558848,442.479167,2017.195116,6874.015755,19.000000,361.790303,253.253212,310.714286,1,6
C-4,4.558848,594.625000,2710.804793,9237.636314,29.000000,318.539183,222.977428,310.714286,1,6
C-4.2,4.558848,988.250000,4505.281206,15352.691338,29.000000,529.403150,370.582205,460.714286,1,4
C-5,4.558848,735.904167,3354.874993,11432.440704,29.000000,394.222093,275.955465,310.714286,1,6
C-5.5,4.558848,703.166667,3205.629717,10923.856102,29.000000,376.684693,263.679285,310.714286,1,6


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-1.7,3.171372,996.625000,3160.668908,18643.467800,20.000000,932.173390,652.521373,769.642857,1,2
C-2.1,3.171372,703.000000,2229.474719,13150.741616,29.000000,453.473849,317.431694,460.714286,1,4
C-2.9,3.171372,988.250000,3134.108665,18486.800003,29.000000,637.475862,446.233104,460.714286,1,4
C-3,3.171372,1200.416667,3806.968152,22455.717518,14.250000,1575.839826,1103.087878,1200.000000,2,3
C-3.3,3.171372,442.479167,1403.266168,8277.281922,19.000000,435.646417,304.952492,310.714286,1,6
C-4,3.171372,594.625000,1885.777248,11123.413561,29.000000,383.565985,268.496189,310.714286,1,6
C-4.2,3.171372,988.250000,3134.108665,18486.800003,29.000000,637.475862,446.233104,460.714286,1,4
C-5,3.171372,735.904167,2333.826082,13766.266786,29.000000,474.698855,332.289198,460.714286,1,4
C-5.5,3.171372,703.166667,2230.003282,13153.859383,29.000000,453.581358,317.506951,460.714286,1,4


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-1.7,1.817686,996.625000,1811.551604,20455.019404,20.000000,1022.750970,715.925679,769.642857,1,2
C-2.1,1.817686,703.000000,1277.833465,14428.575082,29.000000,497.537072,348.275950,460.714286,1,4
C-2.9,1.817686,988.250000,1796.328481,20283.128484,29.000000,699.418224,489.592757,600.000000,1,3
C-3,1.817686,1200.416667,2181.980923,24637.698442,14.250000,1728.961294,1210.272906,1539.285714,2,2
C-3.3,1.817686,442.479167,804.288317,9081.570239,19.000000,477.977381,334.584167,460.714286,1,4
C-4,1.817686,594.625000,1080.841713,12204.255275,29.000000,420.836389,294.585472,310.714286,1,6
C-4.2,1.817686,988.250000,1796.328481,20283.128484,29.000000,699.418224,489.592757,600.000000,1,3
C-5,1.817686,735.904167,1337.642918,15103.909704,29.000000,520.824473,364.577131,460.714286,1,4
C-5.5,1.817686,703.166667,1278.136413,14431.995797,29.000000,497.655027,348.358519,460.714286,1,4


#### East-West Direction

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

In [51]:
# Wall tributary widths and lengths
t_w__CA5 = fi(22, 7)
t_w__CB = l_CA__CB + l_CB__CC / 2
t_w__CC = l_CB__CC / 2 + l_CC__CD
t_w__CX = l_CW__CX + l_CX__CY / 2
t_w__CY = l_CX__CY / 2 + l_CY__CZ
t_w__CY3 = 15.75
t_w__CY6 = 17.25
t_w__CW6 = t_w__CX

t_l__CB__1 = fi(41, 5)
# t_l__CB__2 = fi(23, 5)
t_l__CB__3 = fi(28, 4)
t_l__CB__4 = fi(23, 1)
t_l__CC__1 = fi(77, 4)
t_l__CC__2 = fi(31, 7)
t_l__CC__3 = fi(17, 6)

t_l__CX__1 = fi(13, 10)
t_l__CX__2 = fi(21)
t_l__CX__3 = fi(19, 8)
t_l__CW6__1 = fi(27, 8)
t_l__CY__1 = fi(14, 1)
t_l__CY__2 = fi(16, 4)
t_l__CY__3 = fi(13, 10)
t_l__CY3 = fi(46, 8)
t_l__CY6 = t_l__CY3

In [52]:
# Wall segment lengths
l_CA5 = fi(14, 9)
l_CB__1 = fi(23, 3)
# l_CB__2 = fi(12, 9)
l_CB__3 = fi(9, 11)
l_CB__4 = fi(18, 8)
l_CC__1 = fi(28)
l_CC__2 = fi(19, 4)
l_CC__3 = fi(12, 1)

# l_CX__1 = fi(9, 10)
l_CX__2 = fi(13, 0)
l_CX__3 = fi(11, 0)
l_CW6__1 = fi(19, 1)
l_CY__1 = fi(11, 4)
l_CY__2 = fi(8, 10)
l_CY3 = fi(8, 7)
l_CY6 = fi(12, 9)

In [53]:
flexible_ew.loc["Level 5", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__5)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 1", "tributary weight"] = float(f_p__2)

# Nonparallel irregularity
angle_degrees = 23
angle = math.radians(angle_degrees)
angled_walls = [
    "C-A.5--C-2",
    "C-B--C-2",
    "C-B--C-3.3",
    "C-B--C-4",
    "C-B--C-4.9",
    "C-C--C-2.5",
    "C-C--C-3.6",
    "C-C--C-5.2",
]
wall_indices = flexible_ew.index.get_level_values("Wall").isin(angled_walls)
flexible_ew.loc[wall_indices, "tributary weight"] *= math.cos(angle) + 0.3 * math.sin(angle)

# Shear wall height is for the floor below
flexible_ew.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "C-A.5--C-2": {"trib_width": float(t_w__CA5), "trib_depth": t_l__CB__1, "wall_length": l_CA5},
    "C-B--C-2": {"trib_width": fi(12), "trib_depth": t_l__CB__1, "wall_length": l_CB__1},
    "C-B--C-4": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__3, "wall_length": l_CB__3},
    "C-B--C-4.9": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__4, "wall_length": l_CB__4},
    "C-C--C-2.5": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__1, "wall_length": l_CC__1},
    "C-C--C-3.6": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__2, "wall_length": l_CC__2},
    "C-C--C-5.2": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__3, "wall_length": l_CC__3},
    # "C-X--C-5.8": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__1, "wall_length": l_CX__1},
    "C-X--C-6.5": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__2, "wall_length": l_CX__2},
    "C-X--C-8.5": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__3, "wall_length": l_CX__3},
    "C-W.6--C-7.3": {"trib_width": float(t_w__CW6), "trib_depth": t_l__CW6__1, "wall_length": l_CW6__1},
    # "C-Y--C-5.8": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__1, "wall_length": l_CY__1},
    "C-Y--C-6.5": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__2, "wall_length": l_CY__1},
    "C-Y--C-7.2": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__3, "wall_length": l_CY__2},
    "C-Y.3--C-8": {"trib_width": float(t_w__CY3), "trib_depth": t_l__CY3, "wall_length": l_CY3},
    "C-Y.6--C-8": {"trib_width": float(t_w__CY6), "trib_depth": t_l__CY6, "wall_length": l_CY6},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map({
    key: value["wall_length"] for key, value in wall_tributary.items()
})
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map({
    key: value["trib_width"] for key, value in wall_tributary.items()
})
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map({
    key: value["trib_depth"] for key, value in wall_tributary.items()
})
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-B--C-2", "tributary area"] += 100
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-B--C-4.9", "tributary area"] -= 121.16
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-C--C-5.2", "tributary area"] += 150
# flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-X--C-5.8", "tributary area"] -= 80.25
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-X--C-6.5", "tributary area"] += 380
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-X--C-8.5", "tributary area"] += 58.67
# flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-Y--C-5.8", "tributary area"] += 145
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-Y--C-6.5", "tributary area"] += 480 + 110
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-W.6--C-7.3", "tributary area"] += 30

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]
flexible_ew["aspect ratio factor"] = np.where(
    flexible_ew["aspect ratio"] > 2, 1.25 - 0.125 * flexible_ew["wall height"] / flexible_ew["wall length"], 1
)

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [54]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<=")

<IPython.core.display.Latex object>

In [55]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [56]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [57]:
# Update dataframe
flexible.update(flexible_ew)

In [58]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [59]:
display_table(
    dataframe=flexible_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    position="H",
    caption="Shear wall forces EW - Level 5",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    position="H",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position="H",
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 1",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 1",
    label="tab:shear_wall_forces_level1_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-A.5--C-2,5.030060,935.326389,4704.747468,4704.747468,14.750000,318.965930,223.276151,310.714286,1,6
C-B--C-2,5.030060,597.000000,4959.907520,4959.907520,23.250000,213.329356,149.330549,310.714286,1,6
C-B--C-4,5.030060,949.166667,2115.516111,2115.516111,9.916667,213.329356,149.330549,310.714286,1,6
C-B--C-4.9,5.030060,652.131667,3982.147973,3982.147973,18.666667,213.329356,149.330549,310.714286,1,6
C-W.6--C-7.3,4.847203,956.833333,4637.965191,4637.965191,19.083333,243.037477,170.126234,310.714286,1,6
C-X--C-6.5,4.847203,1083.500000,4728.657028,4728.657028,13.000000,363.742848,254.619994,310.714286,1,6
C-X--C-8.5,4.847203,717.503333,4001.171331,4001.171331,11.000000,363.742848,254.619994,310.714286,1,6
C-C--C-2.5,5.030060,2590.666667,10394.134881,10394.134881,28.000000,371.219103,259.853372,310.714286,1,6
C-C--C-3.6,5.030060,1058.041667,7176.902656,7176.902656,19.333333,371.219103,259.853372,310.714286,1,6


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-A.5--C-2,5.946323,935.326389,5561.752859,10266.500327,14.750000,696.033920,487.223744,600.000000,1,3
C-B--C-2,5.946323,597.000000,5863.392247,10823.299767,23.250000,465.518270,325.862789,460.714286,1,4
C-B--C-4,5.946323,949.166667,2500.873396,4616.389506,9.916667,465.518270,325.862789,460.714286,1,4
C-B--C-4.9,5.946323,652.131667,4707.526392,8689.674365,18.666667,465.518270,325.862789,460.714286,1,4
C-W.6--C-7.3,5.730157,956.833333,5482.805684,10120.770874,19.083333,530.346072,371.242250,460.714286,1,4
C-X--C-6.5,5.730157,1083.500000,5590.017726,10318.674754,13.000000,793.744212,555.620948,600.000000,1,3
C-X--C-8.5,5.730157,717.503333,4730.014999,8731.186330,11.000000,793.744212,555.620948,600.000000,1,3
C-C--C-2.5,5.946323,2590.666667,12287.505287,22681.640168,28.000000,810.058577,567.041004,600.000000,1,3
C-C--C-3.6,5.946323,1058.041667,8484.229841,15661.132497,19.333333,810.058577,567.041004,600.000000,1,3


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-A.5--C-2,4.558848,935.326389,4264.010525,14530.510852,14.750000,985.119380,689.583566,769.642857,1,2
C-B--C-2,4.558848,597.000000,4495.267389,15318.567156,23.250000,658.863103,461.204172,600.000000,1,3
C-B--C-4,4.558848,949.166667,1917.336270,6533.725776,9.916667,658.863103,461.204172,600.000000,1,3
C-B--C-4.9,4.558848,652.131667,3609.103567,12298.777932,18.666667,658.863103,461.204172,600.000000,1,3
C-W.6--C-7.3,4.393121,956.833333,4203.484358,14324.255232,19.083333,750.615995,525.431196,600.000000,1,3
C-X--C-6.5,4.393121,1083.500000,4285.680257,14604.355011,13.000000,1123.411924,786.388347,921.428571,2,4
C-X--C-8.5,4.393121,717.503333,3626.344833,12357.531163,11.000000,1123.411924,786.388347,921.428571,2,4
C-C--C-2.5,4.558848,2590.666667,9420.420720,32102.060888,28.000000,1146.502175,802.551522,921.428571,2,4
C-C--C-3.6,4.558848,1058.041667,6504.576211,22165.708708,19.333333,1146.502175,802.551522,921.428571,2,4


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-A.5--C-2,3.171372,935.326389,2966.268191,17496.779043,14.750000,1186.222308,830.355616,921.428571,2,4
C-B--C-2,3.171372,597.000000,3127.142532,18445.709688,23.250000,793.363858,555.354700,600.000000,1,3
C-B--C-4,3.171372,949.166667,1333.799144,7867.524921,9.916667,793.363858,555.354700,600.000000,1,3
C-B--C-4.9,3.171372,652.131667,2510.680742,14809.458674,18.666667,793.363858,555.354700,600.000000,1,3
C-W.6--C-7.3,3.056084,956.833333,2924.163031,17248.418263,19.083333,903.847245,632.693072,769.642857,1,2
C-X--C-6.5,3.056084,1083.500000,2981.342787,17585.697798,13.000000,1352.745984,946.922189,1200.000000,2,3
C-X--C-8.5,3.056084,717.503333,2522.674666,14880.205829,11.000000,1352.745984,946.922189,1200.000000,2,3
C-C--C-2.5,3.171372,2590.666667,6553.336153,38655.397041,28.000000,1380.549894,966.384926,1200.000000,2,3
C-C--C-3.6,3.171372,1058.041667,4524.922582,26690.631290,19.333333,1380.549894,966.384926,1200.000000,2,3


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-A.5--C-2,1.817686,935.326389,1700.129958,19196.909001,14.750000,1301.485356,911.039749,921.428571,2,4
C-B--C-2,1.817686,597.000000,1792.335810,20238.045498,23.250000,870.453570,609.317499,769.642857,1,2
C-B--C-4,1.817686,949.166667,764.472980,8631.997901,9.916667,870.453570,609.317499,769.642857,1,2
C-B--C-4.9,1.817686,652.131667,1439.007962,16248.466636,18.666667,870.453570,609.317499,769.642857,1,2
C-W.6--C-7.3,1.751608,956.833333,1675.997197,18924.415460,19.083333,991.672426,694.170698,769.642857,1,2
C-X--C-6.5,1.751608,1083.500000,1708.770031,19294.467829,13.000000,1484.189833,1038.932883,1200.000000,2,3
C-X--C-8.5,1.751608,717.503333,1445.882334,16326.088163,11.000000,1484.189833,1038.932883,1200.000000,2,3
C-C--C-2.5,1.817686,2590.666667,3756.074098,42411.471139,28.000000,1514.695398,1060.286778,1200.000000,2,3
C-C--C-3.6,1.817686,1058.041667,2593.479734,29284.111025,19.333333,1514.695398,1060.286778,1200.000000,2,3


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [60]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [61]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_level5} through \autoref{tab:wall_rigidities_ns_level2}.

In [62]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["sheathed sides"] = flexible_ns["sheathed sides"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"]
    * rigid_ns["wall height"]
    / (1000 * rigid_ns["shear stiffness"] * rigid_ns["sheathed sides"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [63]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [64]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [65]:
display_table(
    dataframe=rigid_ns,
    levels="Level 5",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 5",
    label="tab:wall_rigidities_ns_level5",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 1",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 1",
    label="tab:wall_rigidities_ns_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-1.7,5013.083134,0.326149,15370.517593,0.056558
C-2.1,3536.131889,0.184287,19188.204172,0.070606
C-2.9,4970.956385,0.222335,22357.980257,0.082270
C-3,6038.167361,0.518401,11647.687940,0.042859
C-3.3,2225.696574,0.229413,9701.702242,0.035699
C-4,2990.999181,0.169831,17611.602420,0.064805
C-4.2,4970.956385,0.222335,22357.980257,0.082270
C-5,3701.641808,0.188676,19619.073232,0.072191
C-5.5,3536.970232,0.184309,19190.438323,0.070614


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-1.7,10939.337337,0.484158,22594.549030,0.058454
C-2.1,7716.396988,0.295137,26145.127380,0.067640
C-2.9,10847.410132,0.378164,28684.417829,0.074209
C-3,13176.232647,0.642102,20520.477142,0.053088
C-3.3,4856.820638,0.337291,14399.500774,0.037253
C-4,6526.831520,0.263593,24761.043537,0.064059
C-4.2,10847.410132,0.378164,28684.417829,0.074209
C-5,8077.565711,0.304714,26508.644326,0.068580
C-5.5,7718.226385,0.295186,26147.028109,0.067645


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-1.7,15482.798892,0.580401,26676.018646,0.057613
C-2.1,10921.266897,0.380122,28730.924810,0.062051
C-2.9,15352.691338,0.427565,35907.288568,0.077550
C-3,18648.749366,0.638988,29184.800967,0.063031
C-3.3,6874.015755,0.419997,16366.809351,0.035348
C-4,9237.636314,0.335477,27535.857291,0.059470
C-4.2,15352.691338,0.427565,35907.288568,0.077550
C-5,11432.440704,0.393677,29040.122308,0.062719
C-5.5,10923.856102,0.380191,28732.546518,0.062054


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-1.7,18643.467800,0.582500,32005.928786,0.061563
C-2.1,13150.741616,0.379224,34678.036204,0.066702
C-2.9,18486.800003,0.496370,37244.001169,0.071638
C-3,22455.717518,0.680600,32993.998179,0.063463
C-3.3,8277.281922,0.477532,17333.453471,0.033340
C-4,11123.413561,0.385483,28855.798079,0.055503
C-4.2,18486.800003,0.496370,37244.001169,0.071638
C-5,13766.266786,0.392737,35052.129219,0.067422
C-5.5,13153.859383,0.379292,34679.998232,0.066706


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-1.7,20455.019404,0.828973,24675.119488,0.059572
C-2.1,14428.575082,0.531583,27142.662225,0.065529
C-2.9,20283.128484,0.641591,31613.796731,0.076323
C-3,24637.698442,0.886406,27795.034321,0.067104
C-3.3,9081.570239,0.586676,15479.712858,0.037372
C-4,12204.255275,0.539188,22634.520001,0.054645
C-4.2,20283.128484,0.641591,31613.796731,0.076323
C-5,15103.909704,0.551017,27410.985946,0.066177
C-5.5,14431.995797,0.531681,27144.070768,0.065532


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_level5}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [66]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["sheathed sides"] = flexible_ew["sheathed sides"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"]
    * rigid_ew["wall height"]
    / (1000 * rigid_ew["shear stiffness"] * rigid_ew["sheathed sides"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [67]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [68]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [69]:
display_table(
    dataframe=rigid_ew,
    levels="Level 5",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 5",
    label="tab:wall_rigidities_ew_level5",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 1",
    position="H",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 1",
    label="tab:wall_rigidities_ew_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-A.5--C-2,4704.747468,0.429111,10963.934740,0.068976
C-B--C-2,4959.907520,0.277958,17844.076166,0.112260
C-B--C-4,2115.516111,0.436562,4845.860442,0.030486
C-B--C-4.9,3982.147973,0.306922,12974.466736,0.081625
C-W.6--C-7.3,4637.965191,0.326853,14189.750589,0.089270
C-X--C-6.5,4728.657028,0.490154,9647.282380,0.060693
C-X--C-8.5,4001.171331,0.529672,7554.055165,0.047524
C-C--C-2.5,10394.134881,0.379473,27390.994904,0.172321
C-C--C-3.6,7176.902656,0.424775,16895.785184,0.106294


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-A.5--C-2,10266.500327,0.588629,17441.370544,0.070745
C-B--C-2,10823.299767,0.411467,26304.206214,0.106694
C-B--C-4,4616.389506,0.578109,7985.324251,0.032390
C-B--C-4.9,8689.674365,0.441898,19664.420174,0.079762
C-W.6--C-7.3,10120.770874,0.480441,21065.580982,0.085445
C-X--C-6.5,10318.674754,0.674229,15304.406562,0.062077
C-X--C-8.5,8731.186330,0.717062,12176.335764,0.049389
C-C--C-2.5,22681.640168,0.557362,40694.631183,0.165063
C-C--C-3.6,15661.132497,0.606537,25820.574185,0.104732


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-A.5--C-2,14530.510852,0.664511,21866.462829,0.061709
C-B--C-2,15318.567156,0.492633,31095.278801,0.087754
C-B--C-4,6533.725776,0.665439,9818.662710,0.027709
C-B--C-4.9,12298.777932,0.524191,23462.415654,0.066213
C-W.6--C-7.3,14324.255232,0.574051,24952.917134,0.070419
C-X--C-6.5,14604.355011,0.596496,24483.571718,0.069095
C-X--C-8.5,12357.531163,0.641871,19252.369703,0.054332
C-C--C-2.5,32102.060888,0.470389,68245.829583,0.192596
C-C--C-3.6,22165.708708,0.522533,42419.742248,0.119712


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-A.5--C-2,17496.779043,0.588632,29724.477827,0.073181
C-B--C-2,18445.709688,0.570152,32352.292554,0.079651
C-B--C-4,7867.524921,0.747246,10528.701633,0.025921
C-B--C-4.9,14809.458674,0.602492,24580.342909,0.060516
C-W.6--C-7.3,17248.418263,0.576289,29930.130115,0.073687
C-X--C-6.5,17585.697798,0.633070,27778.450713,0.068390
C-X--C-8.5,14880.205829,0.680212,21875.820767,0.053858
C-C--C-2.5,38655.397041,0.502397,76941.886972,0.189429
C-C--C-3.6,26690.631290,0.556607,47952.371311,0.118058


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-A.5--C-2,19196.909001,0.848958,22612.316804,0.071430
C-B--C-2,20238.045498,0.699943,28913.834368,0.091336
C-B--C-4,8631.997901,0.954094,9047.319672,0.028579
C-B--C-4.9,16248.466636,0.746356,21770.406015,0.068770
C-W.6--C-7.3,18924.415460,0.819807,23083.974867,0.072920
C-X--C-6.5,19294.467829,0.920654,20957.351483,0.066202
C-X--C-8.5,16326.088163,0.992177,16454.817119,0.051979
C-C--C-2.5,42411.471139,0.722031,58739.147741,0.185550
C-C--C-3.6,29284.111025,0.804476,36401.493514,0.114988


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines B-1 and B-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The top floor has a higher concentration of load on the N-E corner, but this area is relatively small and is only present on the top floor. We will assume the center of mass is the same as for a uniform floor weight.

All dimensions are measured from an origin at the S-E corner of the building (i.e., the intersection of C-9 and C-Z).

In [70]:
%%render params
X_plan = fi(221, 6)  # ft
Y_plan = fi(115, 6)  # ft
CM_x = fi(102)  # ft
CM_y = fi(43, 3)  # ft

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [71]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [72]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [73]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "C-1.7": {"x": fi(192, 8), "y": 0},
    "C-2.1": {"x": fi(182, 8), "y": 0},
    "C-2.9": {"x": fi(160), "y": 0},
    "C-3": {"x": fi(145, 1), "y": 0},
    "C-3.3": {"x": fi(137, 1), "y": 0},
    "C-4": {"x": fi(118, 9), "y": 0},
    "C-4.2": {"x": fi(120, 4), "y": 0},
    "C-5": {"x": fi(102, 2), "y": 0},
    "C-5.5": {"x": fi(104, 11), "y": 0},
    "C-5.9": {"x": l_C59__C7 + l_C7__C83 + l_C83__C9, "y": 0},
    "C-6": {"x": l_C6__C7 + l_C7__C8 + l_C8__C9, "y": 0},
    "C-7--C-W": {"x": l_C7__C8 + l_C8__C9, "y": 0},
    "C-7--C-Y": {"x": l_C7__C8 + l_C8__C9, "y": 0},
    "C-7.3": {"x": l_C73__C8 + l_C8__C9, "y": 0},
    "C-7.4": {"x": fi(29, 2) + l_C8__C9, "y": 0},
    "C-7.9": {"x": fi(12, 6) + l_C8__C9, "y": 0},
    # "C-8--C-W": {"x": l_C8__C9, "y": 0},
    "C-8--C-W.6": {"x": l_C8__C9, "y": 0},
    "C-8.1": {"x": l_C81__C83 + l_C83__C9, "y": 0},
    "C-8.3": {"x": l_C83__C9, "y": 0},
    # E-W
    "C-A.5--C-2": {"x": 0, "y": fi(82 + 15, 4)},
    "C-B--C-2": {"x": 0, "y": fi(82, 4)},
    "C-B--C-4": {"x": 0, "y": fi(63, 4)},
    "C-B--C-4.9": {"x": 0, "y": fi(55, 8)},
    # "C-X--C-5.8": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-X--C-6.5": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-X--C-8.5": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-W.6--C-7.3": {"x": 0, "y": l_CX__CY + l_CY__CZ + l_CW6__CX},
    "C-C--C-2.5": {"x": 0, "y": fi(69, 11)},
    "C-C--C-3.6": {"x": 0, "y": fi(56, 3)},
    "C-C--C-5.2": {"x": 0, "y": fi(47, 5)},
    # "C-Y--C-5.8": {"x": 0, "y": l_CY__CZ},
    "C-Y--C-6.5": {"x": 0, "y": l_CY__CZ},
    "C-Y--C-7.2": {"x": 0, "y": l_CY__CZ},
    "C-Y.3--C-8": {"x": 0, "y": fi(21, 11)},
    "C-Y.6--C-8": {"x": 0, "y": fi(11, 8)},
}

In [ ]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_x = float(X_plan)
CMx = CM_x
CMxa_plus = CM_x__a__plus
CMxa_minus = CM_x__a__minus
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(
    rigid["ex"] <= 0, 0.05 * plan_dim_x, rigid["ex"]
)  # assign 5% accidental eccentricity to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_y = float(Y_plan)
CMy = CM_y
CMya_plus = CM_y__a__plus
CMya_minus = CM_y__a__minus
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(
    rigid["ey"] <= 0, 0.05 * plan_dim_y, rigid["ey"]
)  # assign 5% accidental eccentricity to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ey"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ex"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_level5} through \autoref{tab:center_of_rigidity_level2}.

In [75]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 5",
    label="tab:center_of_rigidity_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 1",
    label="tab:center_of_rigidity_level1",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-1.7,192.666667,0.000000,105.787402,86.879265,7.287598,0.000000,0.000000,5.775000
C-2.1,182.666667,0.000000,105.787402,76.879265,7.287598,0.000000,0.000000,5.775000
C-2.9,160.000000,0.000000,105.787402,54.212598,7.287598,0.000000,0.000000,5.775000
C-3,145.083333,0.000000,105.787402,39.295931,7.287598,0.000000,0.000000,5.775000
C-3.3,137.083333,0.000000,105.787402,31.295931,7.287598,0.000000,0.000000,5.775000
C-4,118.750000,0.000000,105.787402,12.962598,7.287598,0.000000,0.000000,5.775000
C-4.2,120.333333,0.000000,105.787402,14.545931,7.287598,0.000000,0.000000,5.775000
C-5,102.166667,0.000000,105.787402,3.620735,14.862402,0.000000,0.000000,5.775000
C-5.5,104.916667,0.000000,105.787402,0.870735,14.862402,0.000000,0.000000,5.775000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-1.7,192.666667,0.000000,104.716766,87.949901,8.358234,0.000000,0.000000,5.775000
C-2.1,182.666667,0.000000,104.716766,77.949901,8.358234,0.000000,0.000000,5.775000
C-2.9,160.000000,0.000000,104.716766,55.283234,8.358234,0.000000,0.000000,5.775000
C-3,145.083333,0.000000,104.716766,40.366568,8.358234,0.000000,0.000000,5.775000
C-3.3,137.083333,0.000000,104.716766,32.366568,8.358234,0.000000,0.000000,5.775000
C-4,118.750000,0.000000,104.716766,14.033234,8.358234,0.000000,0.000000,5.775000
C-4.2,120.333333,0.000000,104.716766,15.616568,8.358234,0.000000,0.000000,5.775000
C-5,102.166667,0.000000,104.716766,2.550099,13.791766,0.000000,0.000000,5.775000
C-5.5,104.916667,0.000000,104.716766,0.199901,8.358234,0.000000,0.000000,5.775000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-1.7,192.666667,0.000000,103.641128,89.025538,9.433872,0.000000,0.000000,5.775000
C-2.1,182.666667,0.000000,103.641128,79.025538,9.433872,0.000000,0.000000,5.775000
C-2.9,160.000000,0.000000,103.641128,56.358872,9.433872,0.000000,0.000000,5.775000
C-3,145.083333,0.000000,103.641128,41.442205,9.433872,0.000000,0.000000,5.775000
C-3.3,137.083333,0.000000,103.641128,33.442205,9.433872,0.000000,0.000000,5.775000
C-4,118.750000,0.000000,103.641128,15.108872,9.433872,0.000000,0.000000,5.775000
C-4.2,120.333333,0.000000,103.641128,16.692205,9.433872,0.000000,0.000000,5.775000
C-5,102.166667,0.000000,103.641128,1.474462,12.716128,0.000000,0.000000,5.775000
C-5.5,104.916667,0.000000,103.641128,1.275538,9.433872,0.000000,0.000000,5.775000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-1.7,192.666667,0.000000,103.717347,88.949319,9.357653,0.000000,0.000000,5.775000
C-2.1,182.666667,0.000000,103.717347,78.949319,9.357653,0.000000,0.000000,5.775000
C-2.9,160.000000,0.000000,103.717347,56.282653,9.357653,0.000000,0.000000,5.775000
C-3,145.083333,0.000000,103.717347,41.365986,9.357653,0.000000,0.000000,5.775000
C-3.3,137.083333,0.000000,103.717347,33.365986,9.357653,0.000000,0.000000,5.775000
C-4,118.750000,0.000000,103.717347,15.032653,9.357653,0.000000,0.000000,5.775000
C-4.2,120.333333,0.000000,103.717347,16.615986,9.357653,0.000000,0.000000,5.775000
C-5,102.166667,0.000000,103.717347,1.550681,12.792347,0.000000,0.000000,5.775000
C-5.5,104.916667,0.000000,103.717347,1.199319,9.357653,0.000000,0.000000,5.775000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-1.7,192.666667,0.000000,104.634681,88.031985,8.440319,0.00000,0.000000,5.77500
C-2.1,182.666667,0.000000,104.634681,78.031985,8.440319,0.00000,0.000000,5.77500
C-2.9,160.000000,0.000000,104.634681,55.365319,8.440319,0.00000,0.000000,5.77500
C-3,145.083333,0.000000,104.634681,40.448652,8.440319,0.00000,0.000000,5.77500
C-3.3,137.083333,0.000000,104.634681,32.448652,8.440319,0.00000,0.000000,5.77500
C-4,118.750000,0.000000,104.634681,14.115319,8.440319,0.00000,0.000000,5.77500
C-4.2,120.333333,0.000000,104.634681,15.698652,8.440319,0.00000,0.000000,5.77500
C-5,102.166667,0.000000,104.634681,2.468015,13.709681,0.00000,0.000000,5.77500
C-5.5,104.916667,0.000000,104.634681,0.281985,8.440319,0.00000,0.000000,5.77500


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_level5} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [76]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [77]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 5",
    label="tab:rigid_forces_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position="H",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 1",
    label="tab:rigid_forces_level1",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-1.7,3.833413e+05,483746.779091,86.879265,0.000000,0.056558,679.961452,3754.301547,4434.263000,5013.083134
C-2.1,3.833413e+05,483746.779091,76.879265,0.000000,0.070606,751.144023,4686.784565,5437.928588,3536.131889
C-2.9,3.833413e+05,483746.779091,54.212598,0.000000,0.082270,617.180865,5461.013226,6078.194091,4970.956385
C-3,3.833413e+05,483746.779091,39.295931,0.000000,0.042859,233.059589,2844.987658,3078.047248,6038.167361
C-3.3,3.833413e+05,483746.779091,31.295931,0.000000,0.035699,154.602135,2369.673989,2524.276123,2225.696574
C-4,3.833413e+05,483746.779091,12.962598,0.000000,0.064805,116.244012,4301.694189,4417.938201,2990.999181
C-4.2,3.833413e+05,483746.779091,14.545931,0.000000,0.082270,165.597496,5461.013226,5626.610722,4970.956385
C-5,3.833413e+05,986558.134930,3.620735,0.000000,0.072191,36.170535,4792.025808,4828.196343,3701.641808
C-5.5,3.833413e+05,986558.134930,0.870735,0.000000,0.070614,8.508455,4687.330264,4695.838720,3536.970232


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-1.7,8.365112e+05,1.210694e+06,87.949901,0.000000,0.058454,1519.387818,8467.126752,9986.514570,10939.337337
C-2.1,8.365112e+05,1.210694e+06,77.949901,0.000000,0.067640,1558.245701,9797.677625,11355.923326,7716.396988
C-2.9,8.365112e+05,1.210694e+06,55.283234,0.000000,0.074209,1212.464664,10749.256436,11961.721100,10847.410132
C-3,8.365112e+05,1.210694e+06,40.366568,0.000000,0.053088,633.342868,7689.884881,8323.227750,13176.232647
C-3.3,8.365112e+05,1.210694e+06,32.366568,0.000000,0.037253,356.347477,5396.097885,5752.445362,4856.820638
C-4,8.365112e+05,1.210694e+06,14.033234,0.000000,0.064059,265.678447,9279.003262,9544.681709,6526.831520
C-4.2,8.365112e+05,1.210694e+06,15.616568,0.000000,0.074209,342.500517,10749.256436,11091.756953,10847.410132
C-5,8.365112e+05,1.997743e+06,2.550099,0.000000,0.068580,51.686151,9933.902697,9985.588848,8077.565711
C-5.5,8.365112e+05,1.210694e+06,0.199901,0.000000,0.067645,3.996378,9798.389909,9802.386287,7718.226385


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-1.7,1.183941e+06,1.934052e+06,89.025538,0.000000,0.057613,2116.070901,11811.287792,13927.358693,15482.798892
C-2.1,1.183941e+06,1.934052e+06,79.025538,0.000000,0.062051,2023.073507,12721.134513,14744.208020,10921.266897
C-2.9,1.183941e+06,1.934052e+06,56.358872,0.000000,0.077550,1803.181720,15898.598840,17701.780560,15352.691338
C-3,1.183941e+06,1.934052e+06,41.442205,0.000000,0.063031,1077.690823,12922.096357,13999.787180,18648.749366
C-3.3,1.183941e+06,1.934052e+06,33.442205,0.000000,0.035348,487.700861,7246.699669,7734.400530,6874.015755
C-4,1.183941e+06,1.934052e+06,15.108872,0.000000,0.059470,370.702256,12191.996841,12562.699096,9237.636314
C-4.2,1.183941e+06,1.934052e+06,16.692205,0.000000,0.077550,534.061063,15898.598840,16432.659903,15352.691338
C-5,1.183941e+06,2.606953e+06,1.474462,0.000000,0.062719,38.152810,12858.037275,12896.190084,11432.440704
C-5.5,1.183941e+06,1.934052e+06,1.275538,0.000000,0.062054,32.655942,12721.852553,12754.508495,10923.856102


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-1.7,1.425632e+06,2.310055e+06,88.949319,0.000000,0.061563,2645.624072,15197.521280,17843.145352,18643.467800
C-2.1,1.425632e+06,2.310055e+06,78.949319,0.000000,0.066702,2544.239293,16466.330244,19010.569537,13150.741616
C-2.9,1.425632e+06,2.310055e+06,56.282653,0.000000,0.071638,1947.986574,17684.739102,19632.725677,18486.800003
C-3,1.425632e+06,2.310055e+06,41.365986,0.000000,0.063463,1268.333304,15666.690780,16935.024084,22455.717518
C-3.3,1.425632e+06,2.310055e+06,33.365986,0.000000,0.033340,537.457505,8230.522843,8767.980348,8277.281922
C-4,1.425632e+06,2.310055e+06,15.032653,0.000000,0.055503,403.110195,13701.730335,14104.840530,11123.413561
C-4.2,1.425632e+06,2.310055e+06,16.615986,0.000000,0.071638,575.092254,17684.739102,18259.831356,18486.800003
C-5,1.425632e+06,3.157953e+06,1.550681,0.000000,0.067422,50.511686,16643.962538,16694.474224,13766.266786
C-5.5,1.425632e+06,2.310055e+06,1.199319,0.000000,0.066706,38.651731,16467.261882,16505.913614,13153.859383


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-1.7,1.564158e+06,2.286059e+06,88.031985,0.000000,0.059572,2813.646590,16135.017170,18948.663761,20455.019404
C-2.1,1.564158e+06,2.286059e+06,78.031985,0.000000,0.065529,2743.436325,17748.539020,20491.975345,14428.575082
C-2.9,1.564158e+06,2.286059e+06,55.365319,0.000000,0.076323,2267.170835,20672.205998,22939.376832,20283.128484
C-3,1.564158e+06,2.286059e+06,40.448652,0.000000,0.067104,1456.267149,18175.123985,19631.391134,24637.698442
C-3.3,1.564158e+06,2.286059e+06,32.448652,0.000000,0.037372,650.622900,10122.156972,10772.779872,9081.570239
C-4,1.564158e+06,2.286059e+06,14.115319,0.000000,0.054645,413.839326,14800.672760,15214.512087,12204.255275
C-4.2,1.564158e+06,2.286059e+06,15.698652,0.000000,0.076323,642.848747,20672.205998,21315.054745,20283.128484
C-5,1.564158e+06,3.713266e+06,2.468015,0.000000,0.066177,87.627860,17923.995429,18011.623289,15103.909704
C-5.5,1.564158e+06,2.286059e+06,0.281985,0.000000,0.065532,9.914506,17749.460064,17759.374569,14431.995797


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_level5} through \autoref{tab:shear_walls_level2}.

In [78]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position="H",
    position_float="centering",
    caption="Final shear wall design values - Level 5",
    label="tab:shear_walls_level5",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position="H",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position="H",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position="H",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position="H",
    position_float="centering",
    caption="Final shear wall design values - Level 1",
    label="tab:shear_walls_level1",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-1.7,5013.083134,20.000000,250.654157,175.457910,310.714286,1,6
C-2.1,5437.928588,29.000000,187.514779,131.260345,310.714286,1,6
C-2.9,6078.194091,29.000000,209.592900,146.715030,310.714286,1,6
C-3,6038.167361,14.250000,423.731043,296.611730,310.714286,1,6
C-3.3,2524.276123,19.000000,132.856638,92.999647,310.714286,1,6
C-4,4417.938201,29.000000,152.342697,106.639888,310.714286,1,6
C-4.2,5626.610722,29.000000,194.021059,135.814742,310.714286,1,6
C-5,4828.196343,29.000000,166.489529,116.542670,310.714286,1,6
C-5.5,4695.838720,29.000000,161.925473,113.347831,310.714286,1,6


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-1.7,10939.337337,20.000000,546.966867,382.876807,460.714286,1,4
C-2.1,11355.923326,29.000000,391.583563,274.108494,310.714286,1,6
C-2.9,11961.721100,29.000000,412.473141,288.731199,310.714286,1,6
C-3,13176.232647,14.250000,924.647905,647.253534,769.642857,1,2
C-3.3,5752.445362,19.000000,302.760282,211.932198,310.714286,1,6
C-4,9544.681709,29.000000,329.126955,230.388869,310.714286,1,6
C-4.2,11091.756953,29.000000,382.474378,267.732064,310.714286,1,6
C-5,9985.588848,29.000000,344.330650,241.031455,310.714286,1,6
C-5.5,9802.386287,29.000000,338.013320,236.609324,310.714286,1,6


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-1.7,15482.798892,20.000000,774.139945,541.897961,600.000000,1,3
C-2.1,14744.208020,29.000000,508.420966,355.894676,460.714286,1,4
C-2.9,17701.780560,29.000000,610.406226,427.284358,460.714286,1,4
C-3,18648.749366,14.250000,1308.684166,916.078916,921.428571,2,4
C-3.3,7734.400530,19.000000,407.073712,284.951598,310.714286,1,6
C-4,12562.699096,29.000000,433.196521,303.237564,310.714286,1,6
C-4.2,16432.659903,29.000000,566.643445,396.650411,460.714286,1,4
C-5,12896.190084,29.000000,444.696210,311.287347,460.714286,1,4
C-5.5,12754.508495,29.000000,439.810638,307.867446,310.714286,1,6


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-1.7,18643.467800,20.000000,932.173390,652.521373,769.642857,1,2
C-2.1,19010.569537,29.000000,655.536881,458.875816,460.714286,1,4
C-2.9,19632.725677,29.000000,676.990541,473.893378,600.000000,1,3
C-3,22455.717518,14.250000,1575.839826,1103.087878,1200.000000,2,3
C-3.3,8767.980348,19.000000,461.472650,323.030855,460.714286,1,4
C-4,14104.840530,29.000000,486.373811,340.461668,460.714286,1,4
C-4.2,18486.800003,29.000000,637.475862,446.233104,460.714286,1,4
C-5,16694.474224,29.000000,575.671525,402.970067,460.714286,1,4
C-5.5,16505.913614,29.000000,569.169435,398.418604,460.714286,1,4


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-1.7,20455.019404,20.000000,1022.750970,715.925679,769.642857,1,2
C-2.1,20491.975345,29.000000,706.619839,494.633888,600.000000,1,3
C-2.9,22939.376832,29.000000,791.012994,553.709096,600.000000,1,3
C-3,24637.698442,14.250000,1728.961294,1210.272906,1539.285714,2,2
C-3.3,10772.779872,19.000000,566.988414,396.891890,460.714286,1,4
C-4,15214.512087,29.000000,524.638348,367.246843,460.714286,1,4
C-4.2,21315.054745,29.000000,735.001888,514.501321,600.000000,1,3
C-5,18011.623289,29.000000,621.090458,434.763321,460.714286,1,4
C-5.5,17759.374569,29.000000,612.392227,428.674559,460.714286,1,4


### Shear Wall Axial Forces

#### Tension Forces

The shear forces get converted to overturning forces in the end posts of the shear walls. This overturning moment is counteracted by the dead load tributary to each shear wall according to ASCE 7-16 Sec. 2.4.5. The applicable equation is:

$$
0.6 D - 0.7 E_v + 0.7 E_h \quad\text{(ASCE 7-16 Eq. 2.4.5-10)}
$$

There are no vertical seismic effects, $E_v$, so we will only consider the horizontal seismic effects, $E_h$, and dead loads, $D$.

The overturning moment is:

$$
M_{OT} = V h_{wall}
$$

Using the area dead loads from earlier, along with the tributary areas for each shear wall, we can get the dead load that is applied to each wall. This resists the overturning moment at the centroid of the wall.

$$
M_R = \frac{D l_{wall}}{2}
$$

We will assume that the holddowns are 6" inset from the end posts. The tension caused by horizontal load effects counteracted by dead loads is:

$$
T_{seismic} = \frac{M_{OT} - M_R}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [79]:
shear_walls["line"] = flexible["line"]
shear_walls["wall height"] = rigid["wall height"]
shear_walls["tributary area"] = flexible["tributary area"]
shear_walls["floor OTM"] = shear_walls["shear force demand"] * shear_walls["wall height"]
shear_walls["OTM"] = shear_walls.groupby(level="Wall")["floor OTM"].cumsum()
shear_walls["dead load"] = 0.6 * float(U_floor * 1000) * shear_walls["tributary area"]

# Reduce resisting moment dead load in some walls that definitely are not correct
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-2.1", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-2.9", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-3", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-4.2", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-6", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-7", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-7.3", "dead load"] *= 1 / 8
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-8", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-8.1", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-8.3", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls["line"] == "C-B", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "C-C", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "C-X", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "C-Y", "dead load"] *= 0

shear_walls["RM"] = shear_walls["dead load"] * shear_walls["wall length"] / 2

shear_walls["total moment"] = shear_walls["OTM"] - shear_walls["RM"]
shear_walls["total moment"] = np.where(
    shear_walls["RM"] <= shear_walls["OTM"], shear_walls["OTM"] - shear_walls["RM"], 0
)
wall_arm_shortening_length = 0.5  # feet due to holddown positioning
shear_walls["tension force"] = shear_walls["total moment"] / (shear_walls["wall length"] - wall_arm_shortening_length)
num_bins = 5
shear_walls["binned tension force"] = (
    shear_walls["tension force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else 0))
)

#### Compression Forces

We will calculate the compression forces in a very similar way. However, we will ignore the resisting moment.

The live load, $L$, is $40$ psf, while the snow load, $S$, is $25$ psf and applies only to corridor walls as that is how the roof trusses are supported.

$$
C_{seismic} = \frac{M_{OT}}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [80]:
shear_walls["compression force"] = shear_walls["OTM"] / (shear_walls["wall length"] - wall_arm_shortening_length)
shear_walls["binned compression force"] = (
    shear_walls["compression force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins).map(lambda x: x.right if pd.notna(x) else 0))
    # .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else pd.NA))
)

The tension and compression forces in each wall are shown in the following tables.

In [81]:
column_name_map = {
    "tension force": "$T$ [lbf]",
    "compression force": "$C$ [lbf]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position="H",
    position_float="centering",
    caption="Shear wall axial forces - Level 5",
    label="tab:shear_walls_axial_level5",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position="H",
    position_float="centering",
    caption="Shear wall axial forces - Level 4",
    label="tab:shear_walls_axial_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position="H",
    position_float="centering",
    caption="Shear wall axial forces - Level 3",
    label="tab:shear_walls_axial_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position="H",
    position_float="centering",
    caption="Shear wall axial forces - Level 2",
    label="tab:shear_walls_axial_level2",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position="H",
    position_float="centering",
    caption="Shear wall axial forces - Level 1",
    label="tab:shear_walls_axial_level1",
)

,$T$ [lbf],$C$ [lbf]
Wall,,
C-1.7,0.000000,2699.352457
C-2.1,125.697375,2003.447375
C-2.9,0.000000,2239.334665
C-3,0.000000,4610.964166
C-3.3,0.000000,1432.697259
C-4,0.000000,1627.661442
C-4.2,0.000000,2072.961845
C-5,0.000000,1778.809179
C-5.5,0.000000,1730.045844


,$T$ [lbf],$C$ [lbf]
Wall,,
C-1.7,0.000000,8589.764869
C-2.1,4309.458600,6187.208600
C-2.9,1366.949018,6646.284544
C-3,8141.456597,14672.814551
C-3.3,0.000000,4697.598681
C-4,0.000000,5144.123125
C-4.2,3519.730854,6159.398617
C-5,0.000000,5457.710334
C-5.5,0.000000,5341.451318


,$T$ [lbf],$C$ [lbf]
Wall,,
C-1.7,6193.771965,16926.656580
C-2.1,9741.535239,11619.285239
C-2.9,7888.657645,13167.993171
C-3,22382.319749,28913.677703
C-3.3,4315.793914,9087.393576
C-4,3419.387266,9772.485950
C-4.2,9573.868713,12213.536476
C-5,2346.383216,10208.938259
C-5.5,2527.700062,10040.480764


,$T$ [lbf],$C$ [lbf]
Wall,,
C-1.7,16232.562319,26965.446934
C-2.1,16745.429279,18623.179279
C-2.9,15121.767105,20401.102631
C-3,39530.322217,46061.680172
C-3.3,9292.215192,14063.814854
C-4,8615.907461,14969.006145
C-4.2,16384.795030,19024.462793
C-5,8496.978982,16359.534026
C-5.5,8608.826130,16121.606832


,$T$ [lbf],$C$ [lbf]
Wall,,
C-1.7,30393.729598,41126.614214
C-2.1,26452.154442,28329.904442
C-2.9,25987.787710,31267.123236
C-3,63720.062505,70251.420460
C-3.3,17153.432937,21925.032599
C-4,15822.781607,22175.880291
C-4.2,26481.399909,29121.067672
C-5,17028.800540,24891.355584
C-5.5,17021.161453,24533.942154


#### Holddown Forces Table

With the tension and compression forces calculated, we can construct a table of the forces for each level. We will default to 5 bins for each level to give us options A through E.

However, there is only one table in the plans for all the holddown forces, so these will all get compared together as an aggregate of all buildings and simplified further.

In [82]:
NUM_OPTIONS = 5
LABELS = list("ABCDE")

schedule_rows = []


def assign_schedule(group):
    group = group.copy()

    level = group.index.get_level_values("Level")[0]

    n = len(group)
    group["_bin"] = np.floor(np.arange(n) * min(NUM_OPTIONS, n) / n)

    schedule = (
        group
        .groupby("_bin")
        .agg(
            tension=("tension force", "max"),
            compression=("compression force", "max"),
        )
        .reset_index()
    )

    # convert to kips
    schedule["tension"] /= 1000
    schedule["compression"] /= 1000

    # sort ONLY schedule
    schedule["governing"] = schedule[["tension", "compression"]].max(axis=1)
    schedule = schedule.sort_values("governing").reset_index(drop=True)

    # assign A–E
    schedule["option"] = LABELS[: len(schedule)]

    # IMPORTANT: keep level in schedule
    schedule["Level"] = level

    # map bins → options
    bin_to_option = dict(zip(schedule["_bin"], schedule["option"]))
    group["option"] = group["_bin"].map(bin_to_option)

    schedule_map = schedule.set_index("option")[["tension", "compression"]]

    group["binned tension force"] = group["option"].map(schedule_map["tension"])
    group["binned compression force"] = group["option"].map(schedule_map["compression"])

    schedule_rows.append(schedule[["Level", "option", "tension", "compression"]])

    return group.drop(columns=["_bin"])


shear_walls = shear_walls.groupby(level="Level", group_keys=False).apply(assign_schedule)

schedule_df = (
    pd
    .concat(schedule_rows, ignore_index=True)
    .rename(
        columns={
            "tension": "binned tension force",
            "compression": "binned compression force",
        }
    )
    .set_index(["Level", "option"])
    .sort_index()
)

In [83]:
column_name_map = {
    "binned tension force": "$T$ [kip]",
    "binned compression force": "$C$ [kip]",
}
display_table(
    schedule_df,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    position="H",
    caption="Holddown Schedule",
    label="tab:holddown_schedule",
)

$T$ [kip]  $C$ [kip]
Level   option                      
Level 1 A       32.170146  32.780475
        B       50.284372  60.449893
        C       63.720063  70.251420
        D       65.417346  70.508374
        E       75.372423  75.372423
Level 2 A       20.882814  21.493142
        B       28.961914  39.127435
        C       39.530322  46.061680
        D       42.931536  46.230157
        E       49.241820  49.241820
Level 3 A       12.881272  13.491600
        B       15.234239  24.468499
        C       22.382320  28.913678
        D       26.849865  29.019433
        E       30.308373  30.308373
Level 4 A        6.236250   6.846578
        B        6.341141  12.759283
        C       14.304968  14.304968
        D        8.141457  14.672815
        E       13.625492  14.726482
Level 5 A        1.541224   2.151552
        B        0.330775   3.943266
        C        0.125697   4.610964
        D        4.625879   4.625879
        E        4.281841   4.627829

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [84]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [85]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    position="H",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,371.315313,371.315313,65.034920,65.034920,50.290570,100.581139,65.034920
Level 5,541.375625,912.690938,141.916443,84.179759,73.323366,146.646732,84.179759
Level 4,541.375625,1454.066563,200.858944,74.783465,73.323366,146.646732,74.783465
Level 3,541.375625,1995.442188,241.862422,65.618749,73.323366,146.646732,73.323366
Level 2,551.630000,2547.072188,265.363751,57.470929,74.712208,149.424416,74.712208


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [86]:
# Set up diaphragm dataframe
levels = ["Level 5", "Level 4", "Level 3", "Level 2", "Level 1"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [87]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 35
diaphragms_ns["L_cant"] = 35
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [88]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [89]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 35
diaphragms_ew["depth"] = X_plan
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]

In [90]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [91]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [92]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    position="H",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Level 5 N-S          84179.758613   6.274112       7356.396137   
        E-W          84179.758613   6.274112       3821.718377   
Level 4 N-S          74783.465134   5.573784       6535.262195   
        E-W          74783.465134   5.573784       3395.131415   
Level 3 N-S          73323.366056   5.464960       6407.665402   
        E-W          73323.366056   5.464960       3328.843657   
Level 2 N-S          74712.208215   5.568473       6529.035115   
        E-W          74712.208215   5.568473       3391.896387   
Level 1 N-S              0.000000   0.000000          0.000000   
        E-W              0.000000   0.000000          0.000000   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Level 5 N-S            14712.792275        109.796957        219.593915  
        E-W            48640.052072         17.253808        219.593915  
Level 4 N-S            13070.524390         97.541227        195.082454  
        E-W            43210.763468         15.327907        195.082454  
Level 3 N-S            12815.330804         95.636797        191.273594  
        E-W            42367.101092         15.028640        191.273594  
Level 2 N-S            13058.070229         97.448285        194.896571  
        E-W            43169.590384         15.313302        194.896571  
Level 1 N-S                0.000000          0.000000          0.000000  
        E-W                0.000000          0.000000          0.000000

In [93]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    position="H",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Level 5 N-S              109.796957            189.285714          0.580059   
        E-W               17.253808            189.285714          0.091152   
Level 4 N-S               97.541227            189.285714          0.515312   
        E-W               15.327907            189.285714          0.080978   
Level 3 N-S               95.636797            189.285714          0.505251   
        E-W               15.028640            189.285714          0.079397   
Level 2 N-S               97.448285            189.285714          0.514821   
        E-W               15.313302            189.285714          0.080900   
Level 1 N-S                0.000000            189.285714          0.000000   
        E-W                0.000000            189.285714          0.000000   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Level 5 N-S              219.593915            255.357143          0.859948  
        E-W              219.593915            255.357143          0.859948  
Level 4 N-S              195.082454            214.285714          0.910385  
        E-W              195.082454            214.285714          0.910385  
Level 3 N-S              191.273594            214.285714          0.892610  
        E-W              191.273594            214.285714          0.892610  
Level 2 N-S              194.896571            214.285714          0.909517  
        E-W              194.896571            214.285714          0.909517  
Level 1 N-S                0.000000            189.285714          0.000000  
        E-W                0.000000            189.285714          0.000000

In [94]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'

## Story Drift Determination

The inelastic design story drift is calculated as the difference in deflections between stories at their centers of mass.

We can find the shear wall line that is closest to the center of mass, then take the cumulative force on that line and divide it by the cumulative wall stiffness on that line. This will give us an approximate elastic story drift that will then be converted to inelastic story drift with $C_d$ and $I_e$.

Note that this is a conservative estimation of inelastic story drift due to assuming the deflection of a single shear wall line, rather than the full deflection of the diaphragm as a rigid unit.

\begin{align*}
    \delta_{xe} &= F/k \\
    \delta_x &= \frac{C_d \delta_{xe}}{I_e}
\end{align*}

In [95]:
%%render
C_d
I_e

<IPython.core.display.Latex object>

In [96]:
shear_walls["line"] = flexible["line"]
shear_walls["wall stiffness"] = rigid["wall stiffness"]
shear_walls["cumulative shear demand"] = flexible["cumulative shear demand"]
shear_walls["x"] = rigid["x"]
shear_walls["y"] = rigid["y"]
shear_walls["delta_sw"] = rigid["delta_sw"]


def line_deflection(group, line):
    line_walls = group[group["line"] == line]

    total_stiffness = line_walls["wall stiffness"].sum()
    total_shear = line_walls["cumulative shear demand"].sum()

    return total_shear / total_stiffness


def get_cm_line_deflection(group, CM_x, CM_y, C_d, I_e):
    x_wall = group.loc[(group["x"] - CM_x).abs().idxmin()]

    y_wall = group.loc[(group["y"] - CM_y).abs().idxmin()]

    x_line = x_wall["line"]
    y_line = y_wall["line"]

    x_delta_sw = line_deflection(group, x_line)
    y_delta_sw = line_deflection(group, y_line)

    drift_factor = C_d / I_e

    return pd.Series({
        "x line": x_line,
        "x delta_sw": x_delta_sw,
        "x inelastic drift": x_delta_sw * drift_factor,
        "y line": y_line,
        "y delta_sw": y_delta_sw,
        "y inelastic drift": y_delta_sw * drift_factor,
    })


story_drift = shear_walls.groupby(level="Level", sort=False).apply(
    get_cm_line_deflection,
    CM_x=CM_x,
    CM_y=CM_y,
    C_d=C_d,
    I_e=I_e,
)
story_drift["allowable story drift"] = seismic_loads["floor height"] * 0.02 * 12

In [97]:
column_name_map = {
    "x line": "N-S line",
    "x delta_sw": r"$\delta_{xe,ns}$ [in]",
    "x inelastic drift": r"$\delta_{x,ns}$ [in]",
    "y line": "E-W line",
    "y delta_sw": r"$\delta_{xe,ew}$ [in]",
    "y inelastic drift": r"$\delta_{x,ew}$ [in]",
    "allowable story drift": r"$\Delta_a$ [in]",
}
display_table(
    story_drift,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    position="H",
    caption="Story Drifts",
    label="tab:story_drifts",
)

,N-S line,"$\delta_{xe,ns}$ [in]","$\delta_{x,ns}$ [in]",E-W line,"$\delta_{xe,ew}$ [in]","$\delta_{x,ew}$ [in]",$\Delta_a$ [in]
Level,,,,,,,
Level 5,C-5,0.188676,0.754703,C-W.6,0.326853,1.307413,2.52
Level 4,C-5,0.304714,1.218858,C-W.6,0.480441,1.921764,2.52
Level 3,C-5,0.393677,1.574710,C-W.6,0.574051,2.296205,2.52
Level 2,C-5,0.392737,1.570948,C-W.6,0.576289,2.305158,2.52
Level 1,C-5,0.551017,2.204067,C-W.6,0.819807,3.279230,3.24


In [98]:
# Save dataframes to CSV
shear_walls.to_csv("shear_walls_C.csv")